# Actividad calificable — Corte 2
## Modelo, consulta y limpieza de datos

**Caso:** Registro de servicios de un lavadero de autos.

El objetivo es diseñar un modelo de datos, cargar un dataset en pandas, limpiarlo, comparar el estado antes y después y responder dos preguntas mediante consultas con filtro y agregación.


## 1. ERD (Entity-Relationship Diagram)

El modelo entidad-relación representa la información del lavadero de autos mediante cuatro entidades: `CLIENTE`, `VEHICULO`, `SERVICIO` y `REGISTRO_LAVADO`. Cada entidad contiene atributos que permiten organizar los datos y establecer relaciones entre los clientes, sus vehículos, los servicios disponibles y las atenciones registradas.

```mermaid
erDiagram
    CLIENTE ||--o{ VEHICULO : "posee"
    VEHICULO ||--o{ REGISTRO_LAVADO : "recibe"
    SERVICIO ||--o{ REGISTRO_LAVADO : "se aplica en"

    CLIENTE {
        int id_cliente PK
        string nombre
        string telefono
    }

    VEHICULO {
        string placa PK
        string marca
        string modelo
        int id_cliente FK
    }

    SERVICIO {
        int id_servicio PK
        string tipo_servicio
        float precio_base
    }

    REGISTRO_LAVADO {
        int id_registro PK
        datetime fecha
        string metodo_pago
        float precio_final
        string placa FK
        int id_servicio FK
    }

### Entidades y atributos

| Entidad | Atributo | Descripción |
| :--- | :--- | :--- |
| **CLIENTE** | `id_cliente` (PK) | Identificador único del cliente. |
| **CLIENTE** | `nombre` | Nombre completo del cliente. |
| **CLIENTE** | `telefono` | Teléfono de contacto del cliente. |
| **VEHICULO** | `placa` (PK) | Placa o matrícula única del vehículo. |
| **VEHICULO** | `marca` | Marca del vehículo. |
| **VEHICULO** | `modelo` | Modelo o tipo de vehículo. |
| **VEHICULO** | `id_cliente` (FK) | Código del cliente propietario del vehículo. |
| **SERVICIO** | `id_servicio` (PK) | Identificador único del servicio. |
| **SERVICIO** | `tipo_servicio` | Nombre o tipo de servicio (Lavado general, Polichado, etc.). |
| **SERVICIO** | `precio_base` | Tarifa base registrada para el servicio. |
| **REGISTRO_LAVADO** | `id_registro` (PK) | Identificador único de la transacción de lavado. |
| **REGISTRO_LAVADO** | `fecha` | Fecha y hora en la que se prestó el servicio. |
| **REGISTRO_LAVADO** | `metodo_pago` | Medio de pago utilizado (Efectivo, Transferencia, etc.). |
| **REGISTRO_LAVADO** | `precio_final` | Valor cobrado finalmente al cliente. |
| **REGISTRO_LAVADO** | `placa` (FK) | Placa del vehículo atendido. |
| **REGISTRO_LAVADO** | `id_servicio` (FK) | Identificador del servicio aplicado. |

### Relaciones y cardinalidades

* **CLIENTE — VEHICULO (1:N):** Un cliente puede tener registrados uno o más vehículos en el lavadero. Cada vehículo pertenece a un único cliente.
* **VEHICULO — REGISTRO_LAVADO (1:N):** Un vehículo puede ingresar al lavadero y recibir servicios en una o varias ocasiones. Cada registro de lavado corresponde a un único vehículo.
* **SERVICIO — REGISTRO_LAVADO (1:N):** Un tipo de servicio puede ser contratado en múltiples registros de lavado por distintos clientes. Cada registro de lavado especifica un único servicio prestado.

Las **claves primarias** identifican de forma única cada registro de las entidades, mientras que las **claves foráneas** permiten relacionar las ventas realizadas con los clientes, vehículos y servicios correspondientes.

## 1. Carga del dataset

In [ ]:
import pandas as pd

df = pd.read_csv("../data/servicios_lavadero.csv")
df.head()


## 2. Revisión inicial y conteo de nulos

In [ ]:
print("Filas antes de limpiar:", len(df))
print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos por columna:")
print(df.isnull().sum())

print("\nDuplicados:", df.duplicated().sum())


### Observación inicial
El dataset contiene un valor nulo en `precio`, un registro duplicado y diferencias de formato en las fechas y en el método de pago. También se debe convertir la fecha a un tipo de dato de fecha y el precio a número.


## 3. Limpieza de datos

In [ ]:
# Guardamos el estado inicial para comparar después
filas_antes = len(df)
nulos_antes = int(df.isnull().sum().sum())
duplicados_antes = int(df.duplicated().sum())

# 1. Imputar el precio nulo usando la mediana del mismo tipo de servicio
df["precio"] = df.groupby("tipo_servicio")["precio"].transform(
    lambda x: x.fillna(x.median())
)

# 2. Eliminar registros duplicados
df = df.drop_duplicates().copy()

# 3. Corregir tipos
df["precio"] = pd.to_numeric(df["precio"], errors="coerce")
df["fecha"] = pd.to_datetime(df["fecha"], format="mixed", dayfirst=True, errors="coerce")

# 4. Normalizar formatos de texto
df["cliente"] = df["cliente"].str.strip().str.title()
df["placa"] = df["placa"].str.strip().str.upper()
df["tipo_servicio"] = df["tipo_servicio"].str.strip().str.title()
df["metodo_pago"] = df["metodo_pago"].str.strip().str.title()

# 5. Verificar resultado
print("Nulos después:", int(df.isnull().sum().sum()))
print("Duplicados después:", int(df.duplicated().sum()))
print("Filas después de limpiar:", len(df))
print(df.dtypes)


## 4. Comparación antes y después

In [ ]:
comparacion = pd.DataFrame({
    "Indicador": ["Filas", "Valores nulos", "Duplicados"],
    "Antes": [filas_antes, nulos_antes, duplicados_antes],
    "Después": [
        len(df),
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum())
    ]
})

comparacion


## 5. Consulta 1 — ¿Cuál es el servicio más solicitado?
Se agrupa por tipo de servicio y se cuenta la cantidad de registros.


In [ ]:
consulta_1 = (
    df.groupby("tipo_servicio")
      .size()
      .reset_index(name="cantidad")
      .sort_values("cantidad", ascending=False)
)

consulta_1


### Hallazgo
El resultado permite identificar cuál servicio tiene mayor demanda. Esta información puede ayudar al lavadero a organizar mejor el tiempo de trabajo y los recursos.


## 6. Consulta 2 — ¿Cuánto dinero genera cada tipo de servicio?
Se agrupa por servicio y se suma el precio de las ventas.


In [ ]:
consulta_2 = (
    df.groupby("tipo_servicio")["precio"]
      .agg(["count", "sum", "mean"])
      .reset_index()
      .rename(columns={
          "count": "cantidad",
          "sum": "ingreso_total",
          "mean": "precio_promedio"
      })
      .sort_values("ingreso_total", ascending=False)
)

consulta_2


### Hallazgo
La suma de ingresos permite comparar los servicios no solamente por cantidad de ventas, sino también por el dinero que generan. Un servicio con menos ventas puede generar más ingresos si tiene un precio mayor.


## 7. Conclusión

La limpieza permitió corregir valores nulos, eliminar un duplicado, convertir fechas y precios a los tipos adecuados y normalizar los textos. Las consultas permiten conocer la demanda de los servicios y los ingresos generados por cada uno. Esto facilita tomar decisiones sobre la operación del lavadero.


## 8. Data & cleaning

The dataset contains service records from a car wash business, including the customer, vehicle plate, service type, price, date, and payment method. Before cleaning, the dataset was checked for missing values, duplicated rows, data types, and inconsistent text formats. One missing price was imputed using the median price of the same service type. One duplicated record was removed, and the price and date columns were converted to appropriate data types. Text fields, license plates, and payment methods were normalized to make the information more consistent. The first question asks which service is requested the most, while the second question asks how much revenue each service generates.


## Referencias

1. pandas development team. (s. f.). *pandas documentation: User guide*. https://pandas.pydata.org/docs/user_guide/index.html

2. pandas development team. (s. f.). *pandas.DataFrame.drop_duplicates*. https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html

3. pandas development team. (s. f.). *pandas.DataFrame.groupby*. https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html

4. pandas development team. (s. f.). *pandas.to_datetime*. https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html

5. pandas development team. (s. f.). *pandas.to_numeric*. https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html

6. pandas development team. (s. f.). *pandas.Series.fillna*. https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html